# Phase 3 — train an original arm, resumable across sessions

Runs to the session limit, checkpoints every epoch, and picks up where it left off when
re-run. Re-run this same kernel until `epochs_done` reaches `EPOCHS`.

**Why this is the right thing to train.** The largest measured lever in this competition is
preprocessing, not architecture: `dreaddevelopment` published eight checkpoints of the *same*
`coatnet_rmlp_2_rw_384` at the *same* 384px differing only in corpus preprocessing, and their
gold-gate AUCs span **0.8997 to 0.9214** — a 0.022 spread from slice-span/spacing/FOV alone.
Backbone size moves it the wrong way (`effv2l480`, the largest, scores worst at 0.8716).

**Why the gold gate is the selection metric.** The competition's test labels were produced by
radiologists reading images, not reports. Our own check: ranking blends by report-derived labels
vs by the image-derived gold agrees only at Spearman 0.671 — they disagree about which is best.
The 58 gold studies are noisy (sd ~0.05) but *unbiased*; the 4,349 report labels are precise but
measure the wrong thing. So `best_gold_auc` in the output json is what counts, and it should be
read with its noise in mind.

Fixes carried from Phases 0-2: fp16 (bf16 is 3.13x slower on T4), `--ckpt timm` (the default SSL
backbone was never published), `bs 4 / k 12 / k_eval 16` (the reference `bs 8 / k 12` assumed a
24 GB card), and corpus staged on local disk (the network mount cost ~4.6 of 5.41 s/study).

In [ ]:
TRAIN_PY_B64 = (
    '...'
)

import base64, pathlib, os, shutil, subprocess, sys, time, json, glob
pathlib.Path('/kaggle/working/train_knee.py').write_bytes(base64.b64decode(TRAIN_PY_B64))
EPOCHS = 16           # total across sessions; re-run this kernel until done
SESSION_BUDGET_S = 10.5*3600
def find(name, root='/kaggle/input'):
    for r,d,f in os.walk(root):
        d[:] = [x for x in d if x not in ('train_series','test_series')]
        if name in f: return os.path.join(r,name)
    return None
print('train_knee.py written | EPOCHS target', EPOCHS)

In [ ]:
# Resume state persists via the kernel's own previous output, attached as a kernel_source.
PREV = None
for r,d,f in os.walk('/kaggle/input'):
    d[:] = [x for x in d if x not in ('train_series','test_series')]
    for n in f:
        if n.startswith('resume_p3') and n.endswith('.pt'): PREV = os.path.join(r,n)
if PREV:
    shutil.copyfile(PREV, '/kaggle/working/resume_p3.pt')
    _s = __import__('torch').load('/kaggle/working/resume_p3.pt', map_location='cpu', weights_only=False)
    print(f'RESUMING from epoch {_s["epoch"]} (best gold {_s.get("best",0):.4f})')
    del _s
else:
    print('fresh start (no resume_p3.pt among inputs)')

In [ ]:
# Stage BOTH corpus parts on local disk, merged into one array.
# /tmp has ~1.1 TB (not the 20 GB of /kaggle/working), so all 4,407 studies fit -- worth
# +0.0131 by the author's own measurement of the 3,155 -> 4,349 corpus expansion.
# The script short-circuits to a local single-file corpus, so the two parts are merged here.
import numpy as np
SRC = {n: find(n) for n in ('all_vols.npy','all_masks.npy','all_ids.npy',
                            'extra_vols.npy','extra_masks.npy','extra_ids.npy')}
for k,v in SRC.items(): print(f'  {k:18s} {os.path.getsize(v)/1e9:6.2f} GB')
need = sum(os.path.getsize(v) for v in SRC.values())*1.15
tgt=None
for d in ('/tmp','/kaggle/temp','/kaggle/working'):
    if os.path.isdir(d):
        s=os.statvfs(d)
        print(f'  {d:16s} free {s.f_bavail*s.f_frsize/1e9:7.1f} GB (need {need/1e9:.1f})')
        if s.f_bavail*s.f_frsize > need: tgt=d; break
assert tgt, 'no local volume fits the merged corpus'
STAGE=os.path.join(tgt,'corpus'); os.makedirs(STAGE,exist_ok=True)
MERGED=os.path.join(STAGE,'all_vols.npy')
t0=time.time()
if not os.path.exists(MERGED):
    a=np.load(SRC['all_vols.npy'],mmap_mode='r'); b=np.load(SRC['extra_vols.npy'],mmap_mode='r')
    out=np.lib.format.open_memmap(MERGED,mode='w+',dtype=a.dtype,
                                  shape=(a.shape[0]+b.shape[0],)+a.shape[1:])
    CH=200
    for s_ in range(0,a.shape[0],CH):
        out[s_:s_+CH]=a[s_:s_+CH]; print(f'    part1 {min(s_+CH,a.shape[0])}/{a.shape[0]}  {time.time()-t0:.0f}s',flush=True)
    for s_ in range(0,b.shape[0],CH):
        out[a.shape[0]+s_:a.shape[0]+s_+CH]=b[s_:s_+CH]; print(f'    part2 {min(s_+CH,b.shape[0])}/{b.shape[0]}  {time.time()-t0:.0f}s',flush=True)
    out.flush(); del out,a,b
    np.save(os.path.join(STAGE,'all_masks.npy'),
            np.concatenate([np.load(SRC['all_masks.npy']),np.load(SRC['extra_masks.npy'])],0))
    np.save(os.path.join(STAGE,'all_ids.npy'),
            np.concatenate([np.load(SRC['all_ids.npy'],allow_pickle=True).astype(str),
                            np.load(SRC['extra_ids.npy'],allow_pickle=True).astype(str)]))
for n in ('all_vols.npy','all_masks.npy','all_ids.npy'):
    lk=f'/kaggle/working/{n}'
    if not os.path.exists(lk): os.symlink(os.path.join(STAGE,n),lk)
_v=np.load(MERGED,mmap_mode='r'); print(f'merged corpus {_v.shape} in {(time.time()-t0)/60:.1f} min'); del _v


In [ ]:
LAB=find('labels_llm_soft.parquet'); print('labels:',LAB)
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
cmd=[sys.executable,'/kaggle/working/train_knee.py',
     '--arch','coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k','--res','384',
     '--bs','4','--k','12','--k_eval','16','--grad_ckpt','--ckpt','timm',
     '--labels',LAB,'--epochs',str(EPOCHS),'--tag','p3']
print('$ '+' '.join(cmd[1:]),flush=True)
t0=time.time()
try:
    p=subprocess.run(cmd,cwd='/kaggle/working',capture_output=True,text=True,
                     timeout=SESSION_BUDGET_S)
    print(p.stdout[-6000:])
    if p.returncode: print('STDERR:',p.stderr[-3000:])
except subprocess.TimeoutExpired as e:
    print('session budget reached -- resume_p3.pt holds the state, re-run this kernel')
    if e.stdout: print(e.stdout.decode()[-4000:])
print(f'elapsed {(time.time()-t0)/3600:.2f} h')

In [ ]:
# Report and keep the artifacts the next session needs.
for f in sorted(glob.glob('/kaggle/working/*.pt')+glob.glob('/kaggle/working/*.json')):
    print(f'  {os.path.basename(f):28s} {os.path.getsize(f)/1e6:8.1f} MB')
j='/kaggle/working/raptor_ft_p3.json'
if os.path.exists(j):
    d=json.load(open(j))
    print(f"\n  epochs_done {d.get('epochs_done')}/{EPOCHS} | best gold AUC {d.get('best_gold_auc'):.4f}")
    print('  published reference points on the same 58-study gate:')
    print('    0.9214 raptor_ft_coatnet_v5_full_swa  (best public, used by v7)')
    print('    0.8991 best public label extractor')
    for t,v in (d.get('aucs') or {}).items(): print(f'    {t:18s} {v:.4f}')
# remove the staged corpus so it is not saved as output
import shutil as _sh
for n in ('all_vols.npy','all_masks.npy','all_ids.npy'):
    p_=f'/kaggle/working/{n}'
    if os.path.islink(p_): os.unlink(p_)
print('\nre-run this kernel to continue from resume_p3.pt')